In [1]:
import sys

sys.path.insert(0, "/home/hgf_hmgu/hgf_gib4562/tdmpc2/tdmpc2")
from tdmpc2 import TDMPC2

In [2]:
%load_ext autoreload
%autoreload 2


In [3]:
import torch
from pathlib import Path
import sys
import os

# os.environ['MUJOCO_GL'] = 'egl'
# os.environ['LAZY_LEGACY_OP'] = '0'

from common.activation_patcher import ActivationPatcher
from common.parser import parse_cfg
from envs import make_env

# --- Load base config from file and override with our notebook settings ---

from omegaconf import OmegaConf, DictConfig


/home/hgf_hmgu/hgf_gib4562/miniconda3/envs/tdmpc2/lib/python3.9/site-packages/glfw/__init__.py:914: GLFWError: (65544) b'X11: The DISPLAY environment variable is missing'
  warnings.warn(message, GLFWError)


In [4]:
# --- Utility: Run one episode and return total reward ---
def run_episode(env, agent, max_steps=1000):
    obs = env.reset()
    done = False
    total_reward = 0
    t = 0

    while not done and t < max_steps:
        action = agent.act(obs, t0=(t == 0), eval_mode=True)
        obs, reward, done, info = env.step(action)
        total_reward += reward
        t += 1

    return total_reward

In [5]:
from hydra import initialize, compose
import os

# Get the absolute path and convert to relative path
abs_config_dir = "/home/hgf_hmgu/hgf_gib4562/tdmpc2/tdmpc2"
# Change to the parent directory to use relative path
os.chdir(os.path.dirname(abs_config_dir))
config_dir = "tdmpc2"

override_cfg = dict(
    task='cartpole-swingup',
    checkpoint=
    '/home/hgf_hmgu/hgf_gib4562/tdmpc2/tdmpc2/ckpts/cartpole-swingup-3.pt',
    obs='state',
    seed=1,
    compile=False,
    mpc=True,
    multitask=False,
    model_size=5,
)

with initialize(config_path=config_dir, version_base=None):
    # config_name is the base name of the YAML file (without .yaml extension)
    cfg = compose(config_name="config")
    # Now 'cfg' is your Hydra config object and can be used as needed

    # Merge overrides onto the base config
    cfg = OmegaConf.merge(cfg, override_cfg)

    cfg = parse_cfg(cfg)

In [7]:
print("=" * 60)
print("ActivationPatcher Example (notebook, with merged config)")
print("=" * 60)

# --- Initialize env and agent ---
print("\n1. Loading environment and agent...")
env = make_env(cfg)
agent = TDMPC2(cfg)

if Path(cfg.checkpoint).exists():
    print(f"   Loading checkpoint: {cfg.checkpoint}")
    agent.load(cfg.checkpoint)
else:
    print(f"   Warning: Checkpoint not found: {cfg.checkpoint}")
    print("   Using untrained agent")


ActivationPatcher Example (notebook, with merged config)

1. Loading environment and agent...
Episode length: 500
Discount factor: 0.99
   Loading checkpoint: /home/hgf_hmgu/hgf_gib4562/tdmpc2/tdmpc2/ckpts/cartpole-swingup-3.pt


In [8]:
# --- Baseline run ---
print("\n3. Running baseline (no intervention)...")
baseline_reward = run_episode(env, agent)
print(f"   Baseline reward: {baseline_reward:.2f}")



3. Running baseline (no intervention)...
   Baseline reward: 882.50


In [17]:
# --- Create patcher ---
print("\n2. Creating intervention patcher...")
patcher = ActivationPatcher(agent.model, modules_to_hook=['encoder_output'])
#patcher.remove_hooks()


2. Creating intervention patcher...
Setting up hooks for encoder_output
ActivationPatcher initialized with hooks on:
  - encoder_output
Removed all hooks


In [18]:
agent.model._encoder['state']._forward_hooks


OrderedDict([(1,
              <function common.activation_patcher.ActivationPatcher._make_hook.<locals>.hook(module, input, output)>)])

In [19]:
# --- Detect latent dims ---
print("\n4. Detecting latent dimensions...")
patcher.enable()
obs = env.reset()
_ = agent.act(obs, t0=True, eval_mode=True)
encoder_activation = patcher.get_activation('encoder_output')
n_dims = encoder_activation.shape[-1]
print(f"   Latent dimensions: {n_dims}")


4. Detecting latent dimensions...
Interventions enabled
Hook called for encoder_output with output shape: torch.Size([1, 512])
   Latent dimensions: 512


In [ ]:
# --- Ablate dim 0 ---
print("\n5. Running with dimension 0 ablated...")
patcher.clear_interventions()
patcher.add_ablation('encoder_output', dims=[0])

In [ ]:
_ = agent.act(obs, t0=True, eval_mode=True)

Hook called for encoder_output with output shape: torch.Size([1, 512])


In [31]:
# --- Ablate dim 0 ---
print("\n5. Running with dimension 0 ablated...")
patcher.clear_interventions()
patcher.add_ablation('encoder_output', dims=[i for i in range(500)])
ablation_reward = run_episode(env, agent)
print(f"   Ablation reward: {ablation_reward:.2f}")
print(f"   Impact: {baseline_reward - ablation_reward:.2f}")

# # --- Ablate multiple dims ---
# print("\n6. Running with dimensions [0, 5, 10] ablated...")
# patcher.clear_interventions()
# patcher.add_ablation('encoder_output', dims=[0, 5, 10])
# multi_ablation_reward = run_episode(env, agent)
# print(f"   Multi-ablation reward: {multi_ablation_reward:.2f}")
# print(f"   Impact: {baseline_reward - multi_ablation_reward:.2f}")

# # --- Add noise intervention ---
# print("\n7. Running with noise added to encoder...")
# patcher.clear_interventions()
# patcher.add_noise('encoder_output', scale=0.1)
# noise_reward = run_episode(env, agent)
# print(f"   Noise reward: {noise_reward:.2f}")
# print(f"   Impact: {baseline_reward - noise_reward:.2f}")

# # --- Summary/Results ---
# print("\n" + "=" * 60)
# print("Summary")
# print("=" * 60)
# print(f"Baseline:              {baseline_reward:.2f}")
# print(
#     f"Ablate dim 0:          {ablation_reward:.2f} (Δ = {baseline_reward - ablation_reward:+.2f})"
# )
# print(
#     f"Ablate dims [0,5,10]:  {multi_ablation_reward:.2f} (Δ = {baseline_reward - multi_ablation_reward:+.2f})"
# )
# print(
#     f"Add noise (σ=0.1):     {noise_reward:.2f} (Δ = {baseline_reward - noise_reward:+.2f})"
# )
# print("=" * 60)

# # --- Teardown (optional in notebook, but disables hooks/cleanup) ---
# patcher.remove_hooks()
# print("\n✓ Example complete!")



5. Running with dimension 0 ablated...
Cleared all interventions
Added intervention at: encoder_output
  Ablating dimensions: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 19